In [0]:
import requests
from pyspark.sql.functions import current_timestamp

BASE_URL = "https://test-api-2-r15g.onrender.com"

# -----------------------
# Routes
# -----------------------
crm_routes = ["customers", "products", "sales"]
erp_routes = ["erp_customers", "erp_locations", "erp_categories"]

# -----------------------
# Function to ingest data
# -----------------------
def ingest_api_data(route, table_prefix):
    url = f"{BASE_URL}/api/{route}"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        records = response.json()

        if not records:
            print(f"No data found for {route}")
            return

        # Create Spark DataFrame
        df = spark.createDataFrame(records)

        # Add ingestion timestamp
        df = df.withColumn("ingestion_time", current_timestamp())


        # Table naming logic
        table_name = f"sqldatawarehouse.bronze.{table_prefix}_{route}"

        # Write to Delta table
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        print(f"Successfully saved to: {table_name}")

    except requests.exceptions.RequestException as e:
        print(f"API error for {route}: {e}")

    except Exception as e:
        print(f"Processing error for {route}: {e}")


# -----------------------
# Run CRM ingestion
# -----------------------
for route in crm_routes:
    ingest_api_data(route, "crm")


# -----------------------
# Run ERP ingestion
# -----------------------
for route in erp_routes:
    ingest_api_data(route, "erp")

In [0]:
%sql
select count(*) from sqldatawarehouse.bronze.crm_customers;